In [2]:
%load_ext autoreload
%autoreload 2

import os
import numpy as np
import pandas as pd
import utils.behavioral_utils as behavioral_utils
import utils.information_utils as information_utils
import utils.visualization_utils as visualization_utils
import utils.pseudo_classifier_utils as pseudo_classifier_utils
import utils.classifier_utils as classifier_utils

import utils.io_utils as io_utils

import utils.glm_utils as glm_utils
from matplotlib import pyplot as plt
import matplotlib
import utils.spike_utils as spike_utils
import utils.subspace_utils as subspace_utils
from trial_splitters.condition_trial_splitter import ConditionTrialSplitter 
from utils.session_data import SessionData
from constants.behavioral_constants import *
from constants.decoding_constants import *
import seaborn as sns
from scripts.pseudo_decoding.belief_partitions.belief_partition_configs import *
import scripts.pseudo_decoding.belief_partitions.belief_partitions_io as belief_partitions_io

import scipy
import argparse
import copy
import plotly.express as px
from scripts.anova_analysis.anova_configs import *



In [16]:
region_units_w_label = pd.read_csv("/data/patrick_res/figures/wcst_paper/units/region_units_with_labels.csv")
# drop NaN row
region_units_w_label = region_units_w_label.dropna(subset=["Region (structure level 2)"])
region_units_correct = pd.read_csv("/data/patrick_res/figures/wcst_paper/units/region_units_bl_corrected.csv")

In [17]:
# outer merge
merged = pd.merge(region_units_w_label[["Region (structure level 2)", "Display Name", "Abbreviation"]], region_units_correct, left_on="Region (structure level 2)", right_on="Region", how="outer")

In [19]:
merged

,Region (structure level 2),Display Name,Abbreviation,Region,Subject S Units,Subject B Units Original,Subject B Units Corrected,Subject B Diff,Both Subjects Units Corrected
0,All,NaN,NaN,All,1089.0,343.0,339.0,-4.0,1428.0
1,lateral_prefrontal_cortex (lat_PFC),lateral prefrontal cortex,LPFC,lateral_prefrontal_cortex (lat_PFC),239.0,0.0,0.0,0.0,239.0
2,inferior_temporal_cortex (ITC),inferior temporal cortex,ITC,inferior_temporal_cortex (ITC),185.0,33.0,56.0,23.0,241.0
3,medial_pallium (MPal),hippocampal formation,HC,medial_pallium (MPal),48.0,129.0,83.0,-46.0,131.0
4,amygdala (Amy),amygdala,AMY,amygdala (Amy),73.0,49.0,24.0,-25.0,97.0
5,basal_ganglia (BG),dorsal striatum,DSTR,basal_ganglia (BG),77.0,38.0,34.0,-4.0,111.0
6,anterior_cingulate_gyrus (ACgG),anterior cingulate gyrus,ACG,anterior_cingulate_gyrus (ACgG),85.0,7.0,0.0,-7.0,85.0
7,lateral_and_ventral_pallium (LVPal),claustrum,NaN,lateral_and_ventral_pallium (LVPal),77.0,6.0,7.0,1.0,84.0
8,superior_parietal_lobule (SPL),superior parietal cortex,NaN,superior_parietal_lobule (SPL),61.0,7.0,16.0,9.0,77.0
9,motor_cortex (motor),motor cortex,NaN,motor_cortex (motor),63.0,0.0,0.0,0.0,63.0


In [11]:
merged.to_csv("/data/patrick_res/figures/wcst_paper/units/region_units_bl_corrected_with_labels.csv", index=False)

### do units in the middle_temporal_area (MT) and core_and_belt_areas_of_auditory_cortex show up in significant units?

In [8]:
all_sig_units = []
for sub in ["BL", "SA"]:
    for event in ["StimOnset", "FeedbackOnsetLong"]:
        for var in ["pref", "conf", "response", "choice"]:
            sig_units = pd.read_pickle(f"/data/patrick_res/firing_rates/{sub}/{event}_{var}_99th_window_filter_drift_units.pickle")
            all_sig_units.append(sig_units)
all_sig_units = pd.concat(all_sig_units)

In [10]:
all_sig_units.PseudoUnitID.nunique()

1423

In [12]:
all_sig_units.groupby("structure_level2_cleaned").PseudoUnitID.nunique()

structure_level2_cleaned
:_eexxttrraassttrriiaattee__vviissuuaall__aarreeaass__22--44_VV22--VV44      8
amygdala_Amy                                                               122
anterior_cingulate_gyrus_ACgG                                               92
basal_ganglia_BG                                                           115
extrastriate_visual_areas_2-4_V2-V4                                         25
floor_of_the_lateral_sulcus_floor_of_ls                                      8
inferior_parietal_lobule_IPL                                                51
inferior_temporal_cortex_ITC                                               217
lateral_and_ventral_pallium_LVPal                                           82
lateral_prefrontal_cortex_lat_PFC                                          238
medial_pallium_MPal                                                        175
medial_temporal_lobe_MTL                                                    10
motor_cortex_motor         